# 01 — Metrics Validation

This notebook validates the analytics functions in `api/analytics.py` against **real** data pulled via `yfinance`.

**Network status at time of writing**: network access to Yahoo Finance via `yfinance` **worked** in the build sandbox (confirmed via a direct `yf.Ticker('AAPL').history()` call, and reconfirmed here). A raw, header-less `curl` probe to `query1.finance.yahoo.com` did return HTTP 429 at one point during environment setup, but `yfinance`'s own request handling (which sets appropriate headers/session behavior) succeeded consistently in this environment. See `memory/JOURNAL.md` for the full account.

Because network access worked, this notebook uses **real** data for AAPL, MSFT, and GOOGL, with output values transcribed from an actual run of the pipeline (not fabricated). If you re-run this notebook later and Yahoo Finance is unreachable or rate-limits you, the numbers will differ (or the `yfinance` calls will raise) — that's expected; `api/market_data.py` is what wraps those failures into clean errors for the actual app.

Disclaimer: Educational project. Not investment advice.

In [1]:
import sys
sys.path.insert(0, "..")

import json
import yfinance as yf
from api import analytics

## Pull real data for 3 tickers and run the analytics engine

In [2]:
tickers = ["AAPL", "MSFT", "GOOGL"]
histories = {}
infos = {}

for sym in tickers:
    t = yf.Ticker(sym)
    histories[sym] = t.history(period="1y")
    infos[sym] = t.info
    print(f"{sym}: {len(histories[sym])} rows of daily OHLCV over the trailing 1y period")

AAPL: 251 rows of daily OHLCV over the trailing 1y period
MSFT: 251 rows of daily OHLCV over the trailing 1y period
GOOGL: 251 rows of daily OHLCV over the trailing 1y period


In [3]:
metrics_by_symbol = {}
for sym in tickers:
    m = analytics.compute_price_metrics(histories[sym])
    metrics_by_symbol[sym] = m
    print(f"=== {sym} metrics ===")
    print(json.dumps(m, indent=2, default=str))
    print()

=== AAPL metrics ===
{
  "last_close": 313.3900146484375,
  "sma_50": 295.8526513671875,
  "sma_200": 271.4397592163086,
  "golden_cross": true,
  "volatility_annualized": 0.37625755876554245,
  "max_drawdown": -0.13798524071888174,
  "rsi_14": 59.49283432970273,
  "return_1m": 0.019685099182130017,
  "return_6m": 0.17478197840696197,
  "return_1y": null
}

=== MSFT metrics ===
{
  "last_close": 383.3399963378906,
  "sma_50": 404.9840496826172,
  "sma_200": 441.989762878418,
  "golden_cross": false,
  "volatility_annualized": 0.3734526709693868,
  "max_drawdown": -0.34498391627511604,
  "rsi_14": 45.358000203108425,
  "return_1m": -0.0799913985065891,
  "return_6m": -0.1856886586461748,
  "return_1y": null
}

=== GOOGL metrics ===
{
  "last_close": 361.9200134277344,
  "sma_50": 372.2943292236328,
  "sma_200": 317.52989768981934,
  "golden_cross": true,
  "volatility_annualized": 0.3243616677353609,
  "max_drawdown": -0.20366459999534114,
  "rsi_14": 43.06524731817477,
  "return_1m": -

### Sanity checks on the real output above

- `return_1y` is `null` for all three tickers: the `history(period="1y")` call returns ~251 trading days, which is *not strictly more than* the 252-day window `compute_returns` requires (see `analytics.py::compute_returns` — it needs `len(close) > window`). This is expected, documented behavior, not a bug: a full 252-trading-day 1Y return needs slightly more than exactly one year of history. Confirmed by reading the source: `windows = {"return_1m": 21, "return_6m": 126, "return_1y": 252}` and the check `if len(close) > window`.
- AAPL and GOOGL show `golden_cross: true` (50-day SMA above 200-day SMA) while MSFT shows `false` at the time of this run — consistent with MSFT's negative 1M/6M returns above and AAPL/GOOGL's positive 6M returns.
- RSI values (59.5, 45.4, 43.1) are all comfortably within the neutral 30-70 band, consistent with none of the three showing extreme recent momentum in either direction at the time of this run.
- Max drawdown for MSFT (-34.5%) is much deeper than AAPL (-13.8%) or GOOGL (-20.4%) over the trailing year, consistent with MSFT's SMA50 < SMA200 (death cross) state.


In [4]:
fundamentals_by_symbol = {}
for sym in tickers:
    f = analytics.extract_fundamentals(infos[sym])
    fundamentals_by_symbol[sym] = f
    subset = {k: f[k] for k in ["trailing_pe", "market_cap", "profit_margin", "roe", "sector"]}
    print(f"{sym} fundamentals subset:", subset)

AAPL fundamentals subset: {'trailing_pe': 37.940678, 'market_cap': 4602870628352, 'profit_margin': 0.27152002, 'roe': 1.4147099, 'sector': 'Technology'}
MSFT fundamentals subset: {'trailing_pe': 22.817858, 'market_cap': 2847616270336, 'profit_margin': 0.39341998, 'roe': 0.34013999, 'sector': 'Technology'}
GOOGL fundamentals subset: {'trailing_pe': 27.627481, 'market_cap': 4416355172352, 'profit_margin': 0.37919, 'roe': 0.38884997, 'sector': 'Communication Services'}


**Observation (dead end / gotcha worth noting)**: `yfinance`'s `.info` dict is not perfectly consistent across tickers or over time — e.g. `returnOnEquity` for AAPL (1.41, i.e. 141% ROE) is unusually high due to AAPL's large share buyback program shrinking its book equity; this isn't a bug in `extract_fundamentals`, it's a real (if unusual) reported figure. GOOGL's `sector` comes back as `"Communication Services"` rather than `"Technology"` — this is exactly why the hardcoded `SECTOR_PEER_MAP` peer-matching in `analytics.py` is documented as an approximation (see `docs/DATA.md`), since substring matching against `"tech"` would NOT match GOOGL's actual reported sector string.

## Health score example (AAPL vs MSFT/GOOGL peers)

In [5]:
aapl_metrics = metrics_by_symbol["AAPL"]
aapl_fund = fundamentals_by_symbol["AAPL"]
peer_pes = [fundamentals_by_symbol["MSFT"]["trailing_pe"], fundamentals_by_symbol["GOOGL"]["trailing_pe"]]

health = analytics.compute_health_score(
    target_pe=aapl_fund["trailing_pe"],
    peer_pes=peer_pes,
    last_close=aapl_metrics["last_close"],
    sma_50=aapl_metrics["sma_50"],
    sma_200=aapl_metrics["sma_200"],
    rsi_14=aapl_metrics["rsi_14"],
    profit_margin=aapl_fund["profit_margin"],
    roe=aapl_fund["roe"],
)
print(json.dumps({
    "final_score": health["final_score"],
    "valuation": {"score": round(health["valuation"]["score"], 2), "reason": health["valuation"]["reason"]},
    "momentum": {"score": health["momentum"]["score"]},
    "profitability": {"score": health["profitability"]["score"]},
}, indent=2))

{
  "final_score": 77.4,
  "valuation": {
    "score": 24.79,
    "reason": "Target P/E 37.94 vs peer average 25.22 (ratio 1.50)."
  },
  "momentum": {
    "score": 100.0
  },
  "profitability": {
    "score": 100.0
  }
}


Matches the value returned by the live `GET /api/ticker/AAPL/peers?peers=MSFT,GOOGL` endpoint when tested manually during the build (see `memory/JOURNAL.md`): final score 77.4. Breaking it down using the exact weighted formula from `docs/ARCHITECTURE.md`:

- Valuation (30%): AAPL's P/E (37.94) is 1.50x the MSFT/GOOGL peer average (25.22), which is *more expensive* than peers, so it scores relatively low: `100 - 1.50*50 = 25.0` (24.79 shown above, small rounding from the unrounded ratio).
- Momentum (35%): AAPL was trading above both its SMA50 and SMA200 (trend component 100) with RSI 59.5 inside the neutral 40-60 band (RSI component 100) -> momentum score 100.
- Profitability (35%): AAPL's profit margin (27.2%) and ROE (141.5%) both clear the >=20% bucket for a component score of 100 each -> profitability score 100.

Final = `0.30*24.79 + 0.35*100 + 0.35*100 = 7.44 + 35 + 35 = 77.4` -- matches the printed `final_score` above exactly, confirming the notebook, the live API test during the build, and the documented formula in `docs/ARCHITECTURE.md` are all consistent.

## What a re-run would look like if Yahoo Finance is unreachable

If `yfinance` calls fail (e.g. due to rate-limiting or an outage), `Ticker.history()` typically returns an **empty DataFrame** rather than raising, and `Ticker.info` may return a minimal/near-empty dict. The production app handles this in `api/market_data.py::get_history`/`get_info`, which explicitly check for empty results and raise a typed `MarketDataError` (mapped to a clean 404 JSON response by FastAPI) instead of letting an empty DataFrame silently propagate into `analytics.py` and produce `None`-filled or misleading metrics.

For a conceptual, network-independent illustration, here's the same pipeline run against a small **hand-built synthetic price series** (this always works, since it uses no network):

In [6]:
import pandas as pd

dates = pd.date_range("2024-01-01", periods=10, freq="B")
synthetic_closes = [100, 102, 105, 108, 112, 110, 95, 98, 101, 104]
synthetic_history = pd.DataFrame({
    "Open": synthetic_closes,
    "High": [c * 1.01 for c in synthetic_closes],
    "Low": [c * 0.99 for c in synthetic_closes],
    "Close": synthetic_closes,
    "Volume": [1_000_000] * 10,
}, index=dates)

print("Synthetic SMA(5):", analytics.compute_sma(synthetic_history, 5))
print("Synthetic max drawdown:", analytics.compute_max_drawdown(synthetic_history))

Synthetic SMA(5): 106.0
Synthetic max drawdown: -0.16666666666666663


This synthetic fallback pattern is exactly what `tests/test_analytics.py` uses throughout, so that the full test suite passes with **zero network dependency** regardless of Yahoo Finance's availability at test-run time.

Disclaimer: Educational project. Not investment advice.